# pivot-ai-handball — Notebook Colab

Notebook "thin" : clone le repo, installe les deps, monte Drive et appelle le module Python.
Aucune logique metier ici. Tout est dans `pivot_ai/`.

**Prerequis :** runtime GPU (Edition > Parametres du notebook > Accelerateur materiel = T4 GPU).

## 1. Clone repo + install

In [ ]:
import os, sys

REPO_URL = "https://github.com/REMPLACE_PAR_TON_USERNAME/pivot-ai-handball.git"
BRANCH   = "main"
REPO_DIR = "/content/pivot-ai-handball"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -q -e ".[dev]"

# Reload pour prendre en compte le module installe
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import pivot_ai
print(f"pivot_ai version : {pivot_ai.__version__}")

## 2. Verifier GPU

In [ ]:
import torch
print(f"CUDA dispo : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

## 3. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT  = "/content/drive/MyDrive/PIVOT_AI"
CLIPS_DIR   = f"{DRIVE_ROOT}/raw_clips"
OUTPUTS_DIR = f"{DRIVE_ROOT}/outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

!ls {CLIPS_DIR}

## 4. Choisir un clip et relever les points d'homographie

Sur la frame affichee, survole avec la souris pour relever les pixels des points caracteristiques du terrain (coins, lignes 6m, poteaux de but, sommets des arcs 9m).

Voir `docs/terrain_handball.png` pour le schema avec les noms des points.

In [ ]:
import cv2
import plotly.express as px

CLIP = f"{CLIPS_DIR}/attaque_mhb_stable_58s.mp4"

cap = cv2.VideoCapture(CLIP)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FRAME_REF = total // 2
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_REF)
ret, frame_ref = cap.read()
cap.release()

fig = px.imshow(cv2.cvtColor(frame_ref, cv2.COLOR_BGR2RGB))
fig.update_layout(width=1300, height=730, dragmode='pan',
                  title=f"Frame {FRAME_REF} — survole pour relever les pixels")
fig.show()

In [ ]:
# A remplir avec les pixels releves sur la frame ci-dessus
correspondances = {
    "C":       {"pixel": (879, 319),  "terrain_m": (40, 20)},
    "J":       {"pixel": (974, 355),  "terrain_m": (40, 16)},
    "N":       {"pixel": (1250, 454), "terrain_m": (40, 11.5)},
    "M":       {"pixel": (1426, 524), "terrain_m": (40, 8.5)},
    "I":       {"pixel": (1889, 694), "terrain_m": (40, 4)},
    "L":       {"pixel": (837, 571),  "terrain_m": (34, 10)},
    "O":       {"pixel": (584, 641),  "terrain_m": (31, 10)},
}

## 5. Lancer le pipeline complet

**Attention : la fonction `traiter_match_complet` doit etre implementee par Claude Code avant que cette cellule fonctionne.**

In [ ]:
from pivot_ai.pipeline import traiter_match_complet

resultat = traiter_match_complet(
    chemin_video=CLIP,
    correspondances_homographie=correspondances,
    dossier_sortie=OUTPUTS_DIR,
    subsample=2,
    generer_video_radar=True,
    decouper_actions=True,
)

print("\n=== STATS PAR JOUEUR ===")
print(resultat.stats_joueurs)

print(f"\n=== ACTIONS DETECTEES : {len(resultat.actions_detectees)} ===")
for i, action in enumerate(resultat.actions_detectees):
    print(f"  Action {i+1}: frames {action.frame_debut}-{action.frame_fin} "
          f"({action.duree_s:.1f}s, equipe {action.equipe_en_attaque})")

print(f"\n=== CLIPS DECOUPES : {len(resultat.clips_decoupes)} ===")
for clip in resultat.clips_decoupes:
    print(f"  {clip}")

## 6. Visualiser quelques frames du rendu final

In [ ]:
import matplotlib.pyplot as plt

video_radar = resultat.chemin_video_radar
cap = cv2.VideoCapture(str(video_radar))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fig, axes = plt.subplots(5, 1, figsize=(20, 22))
for ax, idx in zip(axes, [0, total//4, total//2, 3*total//4, total-1]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {idx}/{total}")
        ax.axis('off')
cap.release()
plt.tight_layout()
plt.show()